[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/38.%20The%20Automation%20Investment%20Analysis%20Problem/P38-Tier-2_executed.ipynb)

---

# 38. The Automation Investment Analysis Problem
## Tier 2: A Composite-Score Greedy Heuristic

### Goal

This notebook evaluates automation projects with a multi-criteria score and constructs a portfolio using a budget-constrained greedy heuristic. The score combines risk-adjusted return, strategic value, technology readiness, and implementation ease. Projects are considered in descending score order; a project is accepted whenever its cost fits the remaining budget.

The method is a transparent, fast heuristic. It does not guarantee the portfolio with the greatest expected net present value (NPV), because its ranking score is not the same as portfolio NPV and the selection rule does not reconsider earlier choices.

### Inputs and outputs

Inputs are project costs, scenario-dependent annual benefits, scenario probabilities, project evaluation attributes, a budget, a discount rate, a planning horizon, and scoring weights. The notebook reports project scores, the greedy selection order, selected-project investment and expected portfolio NPV, visual summaries, runtime observations on synthetic instances, and a scoring-weight sensitivity analysis.

### Scope and modeling assumptions

- Each project is either selected in full or not selected; partial investment is not modeled.
- Projects are treated as independent: selecting one does not change another project's cost or benefit.
- Each project's scenario-specific benefit is a constant annual amount received at the end of each year in the planning horizon. There is no terminal value, implementation delay, or benefit interaction.
- The same scenario applies to every selected project, and the listed scenario probabilities sum to one.
- Risk, strategic value, technology readiness, and implementation complexity are supplied as scores rather than estimated from data. Risk, strategic value, readiness, and complexity are on a 0-to-1 scale; larger complexity means more difficult implementation.
- The risk adjustment and score normalization are heuristic choices. The resulting risk-adjusted ROI is a ranking input, not a probability-based risk model or a complete financial valuation.
- This tier implements and analyzes the heuristic. It does not implement an exact optimization baseline, so it makes no empirical comparison with an optimal portfolio.

### Learning sequence

1. Represent projects and scoring weights as data structures.
2. Compute expected annual benefits, risk-adjusted ROI, and composite scores.
3. Rank projects and apply the greedy budget rule.
4. Inspect the portfolio metrics and visualizations.
5. Examine runtime scaling and how alternative scoring weights affect the selected portfolio.

## 1. Model and heuristic

A project score is used to establish a priority order. It is not the same quantity as NPV: the score includes strategic and implementation attributes, whereas NPV measures discounted monetary benefits minus investment cost.

Let \(P\) be the set of projects and \(S\) the set of benefit scenarios. For project \(i\), let \(c_i\) be its cost, \(b_{is}\) its annual benefit in scenario \(s\), \(p_s\) the probability of scenario \(s\), \(r_i\) its risk factor, \(v_i\) its strategic value, \(t_i\) its technology readiness, and \(q_i\) its implementation complexity. The scenario probabilities satisfy \(\sum_{s\in S}p_s=1\).

The expected annual benefit is \(\bar b_i=\sum_{s\in S}p_s b_{is}\). With discount rate \(d\) and planning horizon \(H\), the project's expected NPV and the notebook's risk-adjusted ROI score are

$$\operatorname{NPV}_i=-c_i+\sum_{y=1}^{H}\frac{\bar b_i}{(1+d)^y},\qquad
\operatorname{RAROI}_i=\max\left(0,\frac{\operatorname{NPV}_i(1-0.5r_i)}{c_i}\right).$$

Here, \(y\) indexes years. The risk adjustment multiplies the project's entire NPV, including its initial cost, by \(1-0.5r_i\). This is the heuristic's specified scoring rule; it should not be interpreted as a general financial definition of ROI.

The ROI component is capped and normalized as \(u_i=\min(\operatorname{RAROI}_i/2,1)\). For weights \(w_R,w_V,w_T,w_Q\), the composite score is

$$C_i=w_Ru_i+w_Vv_i+w_Tt_i+w_Q(1-q_i),\qquad
w_R+w_V+w_T+w_Q=1.$$

The term \(1-q_i\) represents implementation ease. Normalization places each score component on a 0-to-1 scale, so the weights express the intended relative importance of the four criteria.

### Greedy selection rule

Sort projects by descending \(C_i\). Starting with remaining budget \(B\), inspect each project in that order and accept it if its cost is no greater than the remaining budget. After accepting project \(i\), reduce the remaining budget by \(c_i\). A project that does not fit is skipped; the algorithm continues to inspect lower-ranked projects.

This rule is easy to follow, but the priority score does not account for cost per unit of score or the opportunity cost of occupying budget. Consequently, feasibility is enforced, but global NPV optimality is not guaranteed. If scores tie, Python's stable sorting preserves the input order among tied projects.

For a selected set \(A\subseteq P\), the portfolio's scenario-specific and expected NPVs are

$$\operatorname{NPV}_s(A)=\sum_{i\in A}\left(-c_i+\sum_{y=1}^{H}\frac{b_{is}}{(1+d)^y}\right),\qquad
\mathbb{E}[\operatorname{NPV}(A)]=\sum_{s\in S}p_s\operatorname{NPV}_s(A).$$

The NPV calculation uses the original scenario benefits; the risk adjustment affects ranking, not the reported portfolio NPV. With \(n=|P|\), \(m=|S|\), and fixed horizon \(H\), scoring and portfolio valuation take \(O(n(m+H))\) time, sorting takes \(O(n\log n)\), and selection takes \(O(n)\). The total is \(O(n(m+H)+n\log n)\); with a fixed number of scenarios and years, this is \(O(n\log n)\). The scored list requires \(O(n)\) additional space.

In the code, `calculate_risk_adjusted_roi` and `calculate_composite_score` implement the project calculations, `greedy_investment_selection` implements the ranking and budget rule, and `calculate_portfolio_npv_heuristic` evaluates the selected portfolio.

## 2. Imports and plotting style

The next cell loads the libraries used for data structures, tabular summaries, numerical calculations, timing, and plots. The algorithm itself is implemented directly in Python; no optimization solver is used.

In [ ]:
from dataclasses import dataclass
from time import time
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported for heuristic investment analysis.")

## 3. Data structures

`ProjectMetrics` stores each project's cost, scenario benefits, and four evaluation attributes. `ScoringWeights` stores the relative importance assigned to the criteria. `HeuristicResult` keeps the selected projects, portfolio measures, elapsed time, and selection order together so later analysis can use one explicit result object.

In [ ]:
@dataclass
class ProjectMetrics:
    """Input data and evaluation attributes for one investment project."""

    id: int
    name: str
    cost: float
    benefits: Dict[str, float]
    risk_factor: float
    strategic_value: float
    tech_readiness: float
    impl_complexity: float


@dataclass
class ScoringWeights:
    """Relative importance assigned to the four score components."""

    roi_weight: float = 0.40
    strategic_weight: float = 0.25
    readiness_weight: float = 0.20
    complexity_weight: float = 0.15


@dataclass
class HeuristicResult:
    """Portfolio and runtime measures returned by the greedy heuristic."""

    selected_projects: List[ProjectMetrics]
    total_investment: float
    portfolio_npv: float
    computation_time: float
    selection_order: List[int]


print("Project, scoring-weight, and result data structures are ready.")

## 4. Sample portfolio and parameters

The sample data below provide three possible annual-benefit scenarios for each project. Scenario probabilities are used to calculate expected benefits and expected portfolio NPV. Costs and benefits are expressed in millions of dollars; the discount rate is annual, and the planning horizon is measured in years. The project attributes are dimensionless scores on a 0-to-1 scale.

In [ ]:
projects_metrics = [
    ProjectMetrics(
        id=1,
        name="Autonomous Crane System",
        cost=20.0,
        benefits={"High": 8.0, "Base": 5.0, "Low": 2.0},
        risk_factor=0.25,
        strategic_value=0.90,
        tech_readiness=0.80,
        impl_complexity=0.70,
    ),
    ProjectMetrics(
        id=2,
        name="AI-Powered Yard Management",
        cost=15.0,
        benefits={"High": 6.0, "Base": 4.0, "Low": 2.0},
        risk_factor=0.15,
        strategic_value=0.85,
        tech_readiness=0.95,
        impl_complexity=0.40,
    ),
    ProjectMetrics(
        id=3,
        name="Integrated Digital Twin Platform",
        cost=18.0,
        benefits={"High": 7.0, "Base": 4.0, "Low": 1.0},
        risk_factor=0.35,
        strategic_value=0.95,
        tech_readiness=0.70,
        impl_complexity=0.80,
    ),
    ProjectMetrics(
        id=4,
        name="Predictive Maintenance AI",
        cost=12.0,
        benefits={"High": 5.0, "Base": 3.5, "Low": 2.0},
        risk_factor=0.10,
        strategic_value=0.70,
        tech_readiness=0.90,
        impl_complexity=0.30,
    ),
    ProjectMetrics(
        id=5,
        name="Autonomous Vehicles",
        cost=25.0,
        benefits={"High": 9.0, "Base": 6.0, "Low": 3.0},
        risk_factor=0.40,
        strategic_value=0.88,
        tech_readiness=0.75,
        impl_complexity=0.85,
    ),
    ProjectMetrics(
        id=6,
        name="Smart Container Tracking",
        cost=8.0,
        benefits={"High": 3.5, "Base": 2.5, "Low": 1.5},
        risk_factor=0.20,
        strategic_value=0.75,
        tech_readiness=0.85,
        impl_complexity=0.35,
    ),
]

scenarios = [
    {"name": "High", "probability": 0.4},
    {"name": "Base", "probability": 0.4},
    {"name": "Low", "probability": 0.2},
]

budget = 50.0
discount_rate = 0.10
planning_horizon = 10

assert np.isclose(sum(scenario["probability"] for scenario in scenarios), 1.0)
assert all(project.cost > 0 for project in projects_metrics)
assert all(
    set(project.benefits) == {scenario["name"] for scenario in scenarios}
    for project in projects_metrics
)

print(f"Defined {len(projects_metrics)} projects with scenario-based benefits.")
print(f"Budget: ${budget:.1f}M; planning horizon: {planning_horizon} years.")

## 5. Project scoring and portfolio valuation

The first function calculates the ranking input from expected annual benefits, discounted over the horizon, and then applies the specified risk adjustment. The composite-score function combines that normalized ROI score with the three supplied project attributes. The portfolio valuation function is separate: it computes scenario-specific NPVs from the unadjusted benefits and averages them using scenario probabilities.

Passing `scenarios` explicitly makes each calculation depend on the scenario data supplied to it, rather than on an implicitly accessed global variable.

In [ ]:
def calculate_risk_adjusted_roi(
    project: ProjectMetrics,
    scenarios: List[Dict[str, float]],
    discount_rate: float,
    horizon: int,
) -> float:
    """Calculate the heuristic's risk-adjusted NPV-to-cost ratio."""
    expected_annual_benefit = sum(
        scenario["probability"] * project.benefits[scenario["name"]]
        for scenario in scenarios
    )

    expected_npv = -project.cost
    for year in range(1, horizon + 1):
        expected_npv += expected_annual_benefit / ((1 + discount_rate) ** year)

    risk_adjusted_npv = expected_npv * (1 - project.risk_factor * 0.5)
    roi = risk_adjusted_npv / project.cost if project.cost > 0 else 0.0

    return max(0.0, roi)


def calculate_composite_score(
    project: ProjectMetrics,
    weights: ScoringWeights,
    scenarios: List[Dict[str, float]],
    discount_rate: float,
    horizon: int,
) -> float:
    """Combine normalized financial and project-attribute scores."""
    weight_total = (
        weights.roi_weight
        + weights.strategic_weight
        + weights.readiness_weight
        + weights.complexity_weight
    )
    if not np.isclose(weight_total, 1.0):
        raise ValueError("Scoring weights must sum to 1.")

    roi_score = calculate_risk_adjusted_roi(
        project, scenarios, discount_rate, horizon
    )
    roi_normalized = min(roi_score / 2.0, 1.0)
    implementation_ease = 1.0 - project.impl_complexity

    return (
        weights.roi_weight * roi_normalized
        + weights.strategic_weight * project.strategic_value
        + weights.readiness_weight * project.tech_readiness
        + weights.complexity_weight * implementation_ease
    )


def calculate_portfolio_npv_heuristic(
    selected_projects: List[ProjectMetrics],
    scenarios: List[Dict[str, float]],
    discount_rate: float,
    horizon: int,
) -> float:
    """Calculate expected NPV for a selected portfolio."""
    total_npv = 0.0

    for scenario in scenarios:
        scenario_npv = 0.0
        for project in selected_projects:
            project_npv = -project.cost
            annual_benefit = project.benefits[scenario["name"]]

            for year in range(1, horizon + 1):
                project_npv += annual_benefit / ((1 + discount_rate) ** year)

            scenario_npv += project_npv

        total_npv += scenario["probability"] * scenario_npv

    return total_npv


print("Project-scoring and portfolio-valuation functions are defined.")

## 6. Greedy budget-constrained selection

The algorithm first scores and sorts every project. It then scans that ranking once, accepting a project only when its cost fits the remaining budget. Skipping a project that does not fit is important: a lower-ranked, less expensive project may still fit. After selection, the function reports total investment and expected portfolio NPV.

The algorithm's elapsed time is measured inside the function. Runtime depends on the machine and execution environment, so it is an observation rather than a guaranteed performance value.

In [ ]:
def greedy_investment_selection(
    projects: List[ProjectMetrics],
    budget: float,
    weights: ScoringWeights,
    scenarios: List[Dict[str, float]],
    discount_rate: float,
    horizon: int,
) -> HeuristicResult:
    """Select projects by composite-score priority under a budget limit."""
    start_time = time()

    scored_projects = []
    for project in projects:
        score = calculate_composite_score(
            project, weights, scenarios, discount_rate, horizon
        )
        scored_projects.append((project, score))

    scored_projects.sort(key=lambda item: item[1], reverse=True)

    selected_projects = []
    remaining_budget = budget
    selection_order = []

    for project, _ in scored_projects:
        if project.cost <= remaining_budget:
            selected_projects.append(project)
            remaining_budget -= project.cost
            selection_order.append(project.id)

    total_investment = sum(project.cost for project in selected_projects)
    portfolio_npv = calculate_portfolio_npv_heuristic(
        selected_projects, scenarios, discount_rate, horizon
    )
    computation_time = time() - start_time

    return HeuristicResult(
        selected_projects=selected_projects,
        total_investment=total_investment,
        portfolio_npv=portfolio_npv,
        computation_time=computation_time,
        selection_order=selection_order,
    )


print("Greedy investment-selection function is defined.")

## 7. Evaluate and rank the projects

This table separates the risk-adjusted ROI input from the composite score. A high value in one component does not by itself determine the ranking: all four components contribute according to their weights. The component summary also reports implementation ease, which is the inverse of implementation complexity.

In [ ]:
weights = ScoringWeights()

project_analysis = []
for project in projects_metrics:
    roi = calculate_risk_adjusted_roi(
        project, scenarios, discount_rate, planning_horizon
    )
    composite_score = calculate_composite_score(
        project, weights, scenarios, discount_rate, planning_horizon
    )

    project_analysis.append(
        {
            "name": project.name,
            "cost": project.cost,
            "risk_adjusted_roi": roi,
            "composite_score": composite_score,
            "strategic_value": project.strategic_value,
            "tech_readiness": project.tech_readiness,
            "impl_complexity": project.impl_complexity,
            "implementation_ease": 1.0 - project.impl_complexity,
        }
    )

analysis_df = pd.DataFrame(project_analysis).sort_values(
    "composite_score", ascending=False
)

print("PROJECT EVALUATION METRICS")
print("=" * 80)
print("\nProject ranking by composite score:")
print(analysis_df.round(3).to_string(index=False))

print("\nScore components by project:")
for _, row in analysis_df.iterrows():
    print(f"\n{row['name']} | Composite score: {row['composite_score']:.3f}")
    print(
        f"Risk-adjusted ROI: {row['risk_adjusted_roi']:.3f} | "
        f"Strategic value: {row['strategic_value']:.3f} | "
        f"Technology readiness: {row['tech_readiness']:.3f} | "
        f"Implementation ease: {row['implementation_ease']:.3f}"
    )

## 8. Run the heuristic and interpret the portfolio

The selection order is the order in which projects are accepted, not a claim that the resulting portfolio is globally optimal. Budget utilization is the selected investment divided by the available budget. Portfolio NPV is the expected monetary value calculated from the original scenario benefits; it is distinct from the composite ranking score.

In [ ]:
result = greedy_investment_selection(
    projects_metrics,
    budget,
    weights,
    scenarios,
    discount_rate,
    planning_horizon,
)

assert result.total_investment <= budget + 1e-9
assert np.isclose(
    result.total_investment,
    sum(project.cost for project in result.selected_projects),
)
assert np.isclose(
    result.portfolio_npv,
    calculate_portfolio_npv_heuristic(
        result.selected_projects, scenarios, discount_rate, planning_horizon
    ),
)

print("\n" + "=" * 70)
print("GREEDY HEURISTIC INVESTMENT-SELECTION RESULTS")
print("=" * 70)
print("\nAlgorithm performance:")
print(f"- Elapsed time: {result.computation_time:.6f} seconds")
print("- Time complexity: O(n(m + H) + n log n), with n projects, m scenarios, and H years")
print(f"- Projects evaluated: {len(projects_metrics)}")

print("\nSelected portfolio (heuristic result):")
print(f"- Selected projects: {len(result.selected_projects)}")
print(f"- Total investment: ${result.total_investment:.1f}M")
print(f"- Budget utilization: {result.total_investment / budget * 100:.1f}%")
print(f"- Remaining budget: ${budget - result.total_investment:.1f}M")
print(f"- Expected portfolio NPV: ${result.portfolio_npv:.2f}M")

project_by_id = {project.id: project for project in projects_metrics}
print("\nSelection order:")
for position, project_id in enumerate(result.selection_order, start=1):
    project = project_by_id[project_id]
    score = calculate_composite_score(
        project, weights, scenarios, discount_rate, planning_horizon
    )
    print(
        f"{position}. {project.name} "
        f"(score: {score:.3f}, cost: ${project.cost:.1f}M)"
    )

print("\nSelected-project details:")
for position, project in enumerate(result.selected_projects, start=1):
    roi = calculate_risk_adjusted_roi(
        project, scenarios, discount_rate, planning_horizon
    )
    score = calculate_composite_score(
        project, weights, scenarios, discount_rate, planning_horizon
    )
    print(f"{position}. {project.name}")
    print(f"   Investment: ${project.cost:.1f}M")
    print(f"   Risk-adjusted ROI score input: {roi:.3f}")
    print(f"   Composite score: {score:.3f}")

## 9. Visual analysis

The plots show score components, cumulative investment and expected NPV in selection order, risk-adjusted ROI against risk, and the selected portfolio's cost composition. In the component chart, the ROI bar is the same capped, normalized value used by the composite-score formula. Project selection is identified explicitly in the scatter-plot legend; bubble size represents project cost.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

component_rows = []
for _, row in analysis_df.iterrows():
    project = next(
        project for project in projects_metrics if project.name == row["name"]
    )
    component_rows.append(
        {
            "name": project.name,
            "Normalized risk-adjusted ROI": min(row["risk_adjusted_roi"] / 2.0, 1.0),
            "Strategic value": project.strategic_value,
            "Technology readiness": project.tech_readiness,
            "Implementation ease": 1.0 - project.impl_complexity,
        }
    )

component_df = pd.DataFrame(component_rows).set_index("name")
component_df.plot(kind="bar", ax=ax1, width=0.8)
ax1.set_xlabel("Project")
ax1.set_ylabel("Normalized component score (0–1)")
ax1.set_title("Composite-score components by project")
ax1.tick_params(axis="x", rotation=45)
ax1.legend(title="Score component", fontsize=8)
ax1.grid(axis="y", alpha=0.3)

cumulative_costs = []
cumulative_npvs = []
project_names = []
cumulative_cost = 0.0
cumulative_npv = 0.0

for project_id in result.selection_order:
    project = project_by_id[project_id]
    cumulative_cost += project.cost
    project_npv = calculate_portfolio_npv_heuristic(
        [project], scenarios, discount_rate, planning_horizon
    )
    cumulative_npv += project_npv
    cumulative_costs.append(cumulative_cost)
    cumulative_npvs.append(cumulative_npv)
    project_names.append(project.name)

ax2_twin = ax2.twinx()
selection_steps = range(1, len(project_names) + 1)
line_cost = ax2.plot(
    selection_steps,
    cumulative_costs,
    "b-o",
    linewidth=2,
    markersize=6,
    label="Cumulative investment",
)
line_npv = ax2_twin.plot(
    selection_steps,
    cumulative_npvs,
    "r-s",
    linewidth=2,
    markersize=6,
    label="Cumulative expected NPV",
)
ax2.axhline(
    y=budget,
    color="blue",
    linestyle="--",
    alpha=0.7,
    label="Budget limit",
)
ax2.set_xlabel("Accepted project, in selection order")
ax2.set_ylabel("Cumulative investment ($M)", color="blue")
ax2_twin.set_ylabel("Cumulative expected NPV ($M)", color="red")
ax2.set_title("Greedy selection process")
ax2.set_xticks(list(selection_steps))
ax2.set_xticklabels(project_names, rotation=45, ha="right")
ax2.grid(True, alpha=0.3)
lines = line_cost + line_npv + ax2.get_lines()[-1:]
ax2.legend(lines, [line.get_label() for line in lines], loc="best", fontsize=8)

selected_ids = {project.id for project in result.selected_projects}
risks = [project.risk_factor for project in projects_metrics]
returns = [
    calculate_risk_adjusted_roi(
        project, scenarios, discount_rate, planning_horizon
    )
    for project in projects_metrics
]
sizes = [project.cost * 3 for project in projects_metrics]
selection_labels = [
    "Selected" if project.id in selected_ids else "Not selected"
    for project in projects_metrics
]
for label, color in [("Selected", "tab:red"), ("Not selected", "lightblue")]:
    indices = [
        index for index, item_label in enumerate(selection_labels)
        if item_label == label
    ]
    if indices:
        ax3.scatter(
            [risks[index] for index in indices],
            [returns[index] for index in indices],
            s=[sizes[index] for index in indices],
            alpha=0.7,
            c=color,
            edgecolors="black",
            label=label,
        )
for project, risk, roi in zip(projects_metrics, risks, returns):
    if project.id in selected_ids:
        ax3.annotate(
            project.name,
            (risk, roi),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )
ax3.set_xlabel("Risk factor (0–1)")
ax3.set_ylabel("Risk-adjusted ROI score input")
ax3.set_title("Risk and risk-adjusted ROI")
ax3.legend(title="Portfolio status")
ax3.grid(True, alpha=0.3)

if result.selected_projects:
    selected_names = [project.name for project in result.selected_projects]
    selected_costs = [project.cost for project in result.selected_projects]
    ax4.pie(
        selected_costs,
        labels=selected_names,
        autopct="%1.1f%%",
        startangle=90,
    )
    ax4.set_title("Selected portfolio composition by investment")
else:
    ax4.text(0.5, 0.5, "No projects selected", ha="center", va="center")
    ax4.set_title("Selected portfolio composition by investment")

plt.tight_layout()
plt.show()
print("Project scores, selection, risk, and portfolio composition are visualized.")

## 10. Runtime scaling and scoring-weight sensitivity

The scaling experiment applies the same heuristic to synthetic project sets of increasing sizes. A fixed random-number generator makes those synthetic inputs reproducible; the sample portfolio above is unchanged. Measured times vary with hardware, software, and system load, so they illustrate the behavior of these runs rather than prove a complexity bound. The theoretical complexity follows from scoring each project, sorting the scores, scanning the ranking, and evaluating the selected portfolio.

The weight sensitivity experiment changes only the relative importance of the score components. It does not alter the sample data, budget, discount rate, or planning horizon. If selected projects change, that indicates the priority ranking is sensitive to the decision-maker's stated preferences. If they do not, the tested weight choices lead to the same selected set for this instance; that does not establish that weights are unimportant in general.

In [ ]:
print("\n" + "=" * 70)
print("HEURISTIC PERFORMANCE ANALYSIS")
print("=" * 70)

problem_sizes = [5, 10, 20, 50, 100, 200]
computation_times = []
rng = np.random.default_rng(42)

print("\nRuntime observations on synthetic project sets:")
for size in problem_sizes:
    test_projects = []
    for project_id in range(size):
        test_projects.append(
            ProjectMetrics(
                id=project_id,
                name=f"Test Project {project_id}",
                cost=float(rng.uniform(5, 50)),
                benefits={
                    "High": float(rng.uniform(2, 15)),
                    "Base": float(rng.uniform(1, 10)),
                    "Low": float(rng.uniform(0.5, 5)),
                },
                risk_factor=float(rng.uniform(0.1, 0.5)),
                strategic_value=float(rng.uniform(0.5, 1.0)),
                tech_readiness=float(rng.uniform(0.6, 1.0)),
                impl_complexity=float(rng.uniform(0.2, 0.9)),
            )
        )

    test_result = greedy_investment_selection(
        test_projects,
        budget * 2,
        weights,
        scenarios,
        discount_rate,
        planning_horizon,
    )
    computation_times.append(test_result.computation_time)
    print(
        f"- {size:3d} projects: "
        f"{test_result.computation_time:.6f} seconds"
    )

print(
    "\nThe measured runtimes are environment-dependent; "
    "the complexity analysis is given in the model section."
)

weight_scenarios = [
    ("ROI-focused", ScoringWeights(0.60, 0.15, 0.15, 0.10)),
    ("Balanced", ScoringWeights(0.40, 0.25, 0.20, 0.15)),
    ("Strategic-focused", ScoringWeights(0.20, 0.45, 0.20, 0.15)),
    ("Readiness-focused", ScoringWeights(0.25, 0.25, 0.40, 0.10)),
]

print("\nScoring-weight sensitivity analysis:")
for scenario_name, scenario_weights in weight_scenarios:
    scenario_result = greedy_investment_selection(
        projects_metrics,
        budget,
        scenario_weights,
        scenarios,
        discount_rate,
        planning_horizon,
    )
    selected_names = [project.name for project in scenario_result.selected_projects]
    print(f"\n{scenario_name} strategy:")
    print(f"- Selected projects: {len(scenario_result.selected_projects)}")
    print(f"- Investment: ${scenario_result.total_investment:.1f}M")
    print(f"- Expected portfolio NPV: ${scenario_result.portfolio_npv:.2f}M")
    print(f"- Projects: {selected_names}")

print("\n" + "=" * 70)
print("HEURISTIC ANALYSIS COMPLETE")
print("=" * 70)

## 11. How to read the results

The project table explains the priority order through the four score components. The selection output shows which projects fit when considered in that order, how much of the budget they use, and the expected NPV of the resulting portfolio. The plots provide complementary views of the same calculations; they do not introduce a separate selection method.

A positive expected portfolio NPV indicates that, under the specified scenario probabilities, benefits discounted over the planning horizon exceed selected-project costs. It does not account for omitted effects such as interactions, implementation timing, changing annual benefits, or uncertainty beyond the listed scenarios. A different set of score weights can produce a different ranking and portfolio, while the NPV calculation continues to use the same scenario-based financial assumptions.

The heuristic is most useful here as a quick, interpretable prioritization procedure. Its output should be understood as the result of the stated scoring and selection rules, not as an optimality certificate.